### Doesn't work ! Temporal Integrated Gradients don't work with this setup, time steps are fed to the model in a expanding window manner, which doesn't work with the fixed window anticipated by the model

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

import torch
import wandb

from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

import lightning as lit
from datetime import datetime

from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule

from machine_intelligence.models.GatedTransformer import GatedTransformer

from machine_intelligence.utilities.viz_setup import figure_setup_grid, VariableNames, insert_line_breaks_in_columns
from machine_intelligence.utilities.viz_setup import figure_setup
from machine_intelligence.interpretability.integrated_gradients import determine_temporal_and_static_attribution


In [ ]:
cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)
model_path = project_root / Path("machine_intelligence/models/trained_models/gtn_final/checkpoints/last.ckpt")
model = GatedTransformer.load_from_checkpoint(model_path, map_location="cpu")
# model = GatedTransformer.load_from_checkpoint(model_path)

In [ ]:

with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4_prev_0.3_imp_none"])

    print(cfg)


In [ ]:

wandb.init(mode="disabled")

In [ ]:

os.environ['PYTHONHASHSEED'] = str(42)
lit.seed_everything(42, workers=True)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

In [ ]:

sepsis_pred_data = SepsisPredDataModule(cfg=cfg)

sepsis_pred_data.setup(stage="fit")
sepsis_pred_data.setup(stage="test")

background_dl = sepsis_pred_data.train_dataloader()
inference_dl = sepsis_pred_data.test_dataloader()


In [ ]:
batch_background = next(iter(background_dl))

# batch = next(iter(inference_dl))
batch_features_ts, batch_features_stat, batch_labels = batch_background

In [ ]:
determine_temporal_and_static_attribution(model=model, batch_to_explain=batch_background, num_samples=1,
                                          considered_features=sepsis_pred_data.considered_features,
                                          considered_features_static=sepsis_pred_data.considered_features_static)